# IN-SO Carbon Intensity: Complete Time-Series Forecasting Study

*Exploration, statistical forecasting, machine learning, residual checks, and clear written commentary*

**Region:** Southern India (`IN-SO`)<br>
**Frequency:** Hourly<br>
**Common period:** 1 January 2017 00:00 UTC to 2 September 2026 15:00 UTC<br>
**Target:** Carbon intensity (gCO₂eq/kWh)<br>
**Contributor:** Shantharam P

This notebook uses the exact same timeline, forecast horizon, model settings, and accuracy measures as the other two regional notebooks.

## 1. Research problem and questions

*What we want to learn and why it matters*

**Main question:** How accurately can the next seven days of hourly carbon intensity be forecast, and which modelling choice works best for this region?

We answer four smaller questions:

1. Is carbon intensity moving up or down over the long term?
2. Does it repeat by hour, day, or season?
3. Does differencing make the series stationary?
4. Which baseline, ARIMA/SARIMA, SARIMAX, Holt-Winters, ML, or DL model gives the smallest unseen error?

A useful forecast can support carbon-aware scheduling. Flexible work can be shifted toward hours or regions with lower expected emissions.

## 2. Setup and reproducibility

*Load the shared analysis functions and interactive plotting tools*

The setup fixes the random seed inside the shared pipeline. Running the notebook again should therefore give the same model comparison, apart from small numerical differences between library versions.

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown, display
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.grid_pulse_analysis import (
    EXPECTED_INDEX, FEATURES, TEST_HOURS, TRAIN_DAYS, VALID_HOURS,
    load_region, run_forecasts, validate_region,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 90)
REGION = "IN-SO"
REGION_NAME = "Southern India"
CONTRIBUTOR = "Shantharam P"
COLORS = {"carbon": "#b91c1c", "renewable": "#15803d", "trend": "#1d4ed8"}
PLOT_CONFIG = {
    "responsive": True,
    "displaylogo": False,
    "scrollZoom": True,
    "modeBarButtonsToRemove": ["lasso2d", "select2d"],
}

def show_chart(fig, height=None):
    '''Render a chart cleanly in Jupyter, VS Code, and presentation mode.'''
    fig.update_layout(
        template="plotly_white",
        autosize=True,
        width=None,
        height=height or fig.layout.height or 520,
        margin=dict(l=70, r=45, t=90, b=85),
        font=dict(family="Arial, sans-serif", size=13, color="#1f2937"),
        title=dict(x=0.01, xanchor="left", font=dict(size=20)),
        hoverlabel=dict(font_size=13),
    )
    fig.update_xaxes(automargin=True, showgrid=True, gridcolor="#e5e7eb")
    fig.update_yaxes(automargin=True, showgrid=True, gridcolor="#e5e7eb", zeroline=False)
    fig.show(config=PLOT_CONFIG)

display(Markdown(
    f"The notebook is ready for **{REGION_NAME}**. "
    "All later sections use one shared and reproducible analysis pipeline."
))

The notebook is ready for **Southern India**. All later sections use one shared and reproducible analysis pipeline.

## 3. Dataset understanding and quality

*Check coverage, units, missing values, duplicates, and the carbon-renewable relationship*

Both signals come from the Electricity Maps hourly past-range endpoints. Carbon intensity is measured in gCO₂eq/kWh. Renewable percentage is the share of electricity attributed to renewable sources.

In [2]:
data = load_region(REGION, ROOT)
checks = pd.DataFrame([validate_region(data)])
description = data.describe().T.rename(index={
    "carbon": "Carbon intensity (gCO₂eq/kWh)",
    "renewable_pct": "Renewable percentage (%)",
})
display(checks)
display(description.round(2))

correlation = data["carbon"].corr(data["renewable_pct"])
display(Markdown(
    f"The dataset has **{len(data):,} complete hourly rows**, "
    f"from **{data.index.min():%d %b %Y %H:%M UTC}** to **{data.index.max():%d %b %Y %H:%M UTC}**. "
    f"There are no gaps, duplicates, or missing target values. Carbon intensity and renewable share have "
    f"correlation **{correlation:.3f}**. The negative sign means cleaner hours usually contain more renewable electricity."
))

,Rows,Start,End,Duplicate timestamps,Missing target values,Missing hours,Extra hours
0,84760,2017-01-01 00:00:00+00:00,2026-09-02 15:00:00+00:00,0,0,0,0


,count,mean,std,min,25%,50%,75%,max
Carbon intensity (gCO₂eq/kWh),84760.0,548.38,84.16,41.0,486.0,555.0,616.0,728.0
Renewable percentage (%),84760.0,27.05,11.58,4.0,17.0,25.0,36.0,82.0


The dataset has **84,760 complete hourly rows**, from **01 Jan 2017 00:00 UTC** to **02 Sep 2026 15:00 UTC**. There are no gaps, duplicates, or missing target values. Carbon intensity and renewable share have correlation **-0.980**. The negative sign means cleaner hours usually contain more renewable electricity.

## 4. Time-series visualization

*See the long-term movement and a recent hourly close-up*

The first panel uses daily averages so nearly ten years remain readable. The second panel keeps hourly detail for the latest two weeks. Drag to zoom, hover for values, or use the range slider.

In [3]:
daily = data.resample("D").mean()
recent = data.iloc[-14 * 24:]
fig = make_subplots(rows=2, cols=1, shared_xaxes=False,
                    subplot_titles=("Daily carbon intensity across the common timeline",
                                    "Latest 14 days: carbon intensity and renewable share"),
                    specs=[[{}], [{"secondary_y": True}]])
fig.add_trace(go.Scatter(x=daily.index, y=daily["carbon"], name="Daily carbon intensity",
                         line=dict(color=COLORS["carbon"], width=1)), row=1, col=1)
fig.add_trace(go.Scatter(x=daily.index, y=daily["carbon"].rolling(365, min_periods=180).mean(),
                         name="365-day average", line=dict(color=COLORS["trend"], width=3)), row=1, col=1)
fig.add_trace(go.Scatter(x=recent.index, y=recent["carbon"], name="Hourly carbon intensity",
                         line=dict(color=COLORS["carbon"])), row=2, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=recent.index, y=recent["renewable_pct"], name="Renewable %",
                         line=dict(color=COLORS["renewable"])), row=2, col=1, secondary_y=True)
fig.update_yaxes(title_text="gCO₂eq/kWh", row=1, col=1)
fig.update_yaxes(title_text="gCO₂eq/kWh", row=2, col=1, secondary_y=False)
fig.update_yaxes(title_text="Renewable %", row=2, col=1, secondary_y=True)
fig.update_layout(height=760, title=f"{REGION}: long-term and recent hourly behaviour", hovermode="x unified")
show_chart(fig)

start_level = daily["carbon"].iloc[:365].mean()
end_level = daily["carbon"].iloc[-365:].mean()
display(Markdown(
    f"The first-year daily average is about **{start_level:.1f}**, while the latest-year average is "
    f"about **{end_level:.1f} gCO₂eq/kWh**. The close-up shows a repeated daily shape and an opposite movement between "
    "renewable share and carbon intensity."
))

The first-year daily average is about **633.4**, while the latest-year average is about **518.4 gCO₂eq/kWh**. The close-up shows a repeated daily shape and an opposite movement between renewable share and carbon intensity.

## 5. Trend and seasonality analysis

*Separate long-term change from the typical daily and yearly patterns*

Annual means summarize the slow trend. Hour-of-day and month-of-year averages show the recurring seasonal shape without forcing a complex model.

In [4]:
annual = data["carbon"].resample("YS").mean()
hourly = data.groupby(data.index.hour)["carbon"].mean()
monthly = data.groupby(data.index.month)["carbon"].mean()
fig = make_subplots(rows=1, cols=3,
                    subplot_titles=("Annual mean", "Typical hour of day (UTC)", "Typical month"))
fig.add_trace(go.Scatter(x=annual.index.year, y=annual, mode="lines+markers",
                         line=dict(color=COLORS["trend"]), name="Annual mean"), row=1, col=1)
fig.add_trace(go.Scatter(x=hourly.index, y=hourly, mode="lines+markers",
                         line=dict(color=COLORS["carbon"]), name="Hourly profile"), row=1, col=2)
fig.add_trace(go.Bar(x=monthly.index, y=monthly, marker_color="#f59e0b", name="Monthly profile"), row=1, col=3)
fig.update_xaxes(title_text="Year", row=1, col=1)
fig.update_xaxes(title_text="Hour", row=1, col=2)
fig.update_xaxes(title_text="Month number", row=1, col=3)
fig.update_layout(height=430, title=f"{REGION}: trend and recurring seasonal patterns", showlegend=False)
show_chart(fig)

annual_change = annual.iloc[-2] - annual.iloc[0] if len(annual) > 2 else annual.iloc[-1] - annual.iloc[0]
daily_swing = hourly.max() - hourly.min()
monthly_swing = monthly.max() - monthly.min()
display(Markdown(
    f"The change from the first complete year to the latest complete year is "
    f"**{annual_change:+.1f} gCO₂eq/kWh**. The average daily swing is **{daily_swing:.1f}**, and the average "
    f"difference between the highest and lowest month is **{monthly_swing:.1f}**. This confirms that both trend and seasonality matter."
))

The change from the first complete year to the latest complete year is **-130.8 gCO₂eq/kWh**. The average daily swing is **94.7**, and the average difference between the highest and lowest month is **125.1**. This confirms that both trend and seasonality matter.

## 6. Stationarity analysis

*Test whether the level and variability stay stable over time*

ADF tests the null hypothesis of a unit root. KPSS tests the opposite null of level stationarity. We use both because agreement is easier to trust. A lag-24 difference means “this hour minus the same hour yesterday.”

In [5]:
def stationarity_row(series, label):
    clean = series.dropna()
    adf_result = adfuller(clean, autolag="AIC")
    kpss_result = kpss(clean, regression="c", nlags="auto")
    return {"Series": label, "ADF statistic": adf_result[0], "ADF p-value": adf_result[1],
            "KPSS statistic": kpss_result[0], "KPSS p-value": kpss_result[1]}

difference_24 = data["carbon"].diff(24).dropna()
stationarity = pd.DataFrame([
    stationarity_row(data["carbon"], "Raw carbon intensity"),
    stationarity_row(difference_24, "Lag-24 differenced carbon intensity"),
])
display(stationarity.round(5))

rolling = data["carbon"].rolling(24 * 30).agg(["mean", "std"]).resample("D").last()
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=("30-day rolling mean", "30-day rolling standard deviation"))
fig.add_trace(go.Scatter(x=rolling.index, y=rolling["mean"], line=dict(color=COLORS["trend"]),
                         name="Rolling mean"), row=1, col=1)
fig.add_trace(go.Scatter(x=rolling.index, y=rolling["std"], line=dict(color="#7c3aed"),
                         name="Rolling standard deviation"), row=2, col=1)
fig.update_layout(height=620, title=f"{REGION}: rolling stability checks", showlegend=False)
show_chart(fig)

raw_kpss = stationarity.loc[0, "KPSS p-value"]
diff_adf = stationarity.loc[1, "ADF p-value"]
diff_kpss = stationarity.loc[1, "KPSS p-value"]
display(Markdown(
    f"The raw KPSS p-value is **{raw_kpss:.4f}**. After lag-24 differencing, "
    f"ADF p = **{diff_adf:.4f}** and KPSS p = **{diff_kpss:.4f}**. "
    "The differenced series is the safer stationary input when ADF is below 0.05 and KPSS is at or above 0.05."
))

,Series,ADF statistic,ADF p-value,KPSS statistic,KPSS p-value
0,Raw carbon intensity,-5.78059,0.0,14.22223,0.01
1,Lag-24 differenced carbon intensity,-34.92074,0.0,0.01295,0.10


The raw KPSS p-value is **0.0100**. After lag-24 differencing, ADF p = **0.0000** and KPSS p = **0.1000**. The differenced series is the safer stationary input when ADF is below 0.05 and KPSS is at or above 0.05.

## 7. ACF and PACF analysis

*Measure how strongly the present depends on earlier hours*

The charts use the latest year and 72 lags. This is enough to show three daily cycles while keeping the calculation responsive.

In [6]:
sample = data["carbon"].iloc[-365 * 24:]
sample_diff = sample.diff(24).dropna()
lag_count = 72
raw_acf, raw_pacf = acf(sample, nlags=lag_count, fft=True), pacf(sample, nlags=lag_count, method="ywm")
diff_acf, diff_pacf = acf(sample_diff, nlags=lag_count, fft=True), pacf(sample_diff, nlags=lag_count, method="ywm")
lags = np.arange(lag_count + 1)
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=("Raw ACF", "Raw PACF", "Lag-24 difference ACF", "Lag-24 difference PACF"))
for values, row, col, color in [
    (raw_acf, 1, 1, "#2563eb"), (raw_pacf, 1, 2, "#7c3aed"),
    (diff_acf, 2, 1, "#0891b2"), (diff_pacf, 2, 2, "#ea580c")]:
    fig.add_trace(go.Bar(x=lags, y=values, marker_color=color, showlegend=False), row=row, col=col)
fig.update_layout(height=700, title=f"{REGION}: autocorrelation before and after daily differencing")
show_chart(fig)

display(Markdown(
    f"Raw correlation at lag 24 is **{raw_acf[24]:.3f}**. After daily differencing it becomes "
    f"**{diff_acf[24]:.3f}**. Strong lag-24 structure supports seasonal models, while the faster decay after differencing "
    "supports ARIMA-style short-memory modelling on the transformed series."
))

Raw correlation at lag 24 is **0.969**. After daily differencing it becomes **-0.109**. Strong lag-24 structure supports seasonal models, while the faster decay after differencing supports ARIMA-style short-memory modelling on the transformed series.

## 8. Initial findings and research hypothesis

*Turn the exploratory evidence into a testable forecast expectation*

The series has long-term movement, a repeating 24-hour pattern, and strong short-lag dependence. Renewable share moves in the opposite direction to carbon intensity.

**Hypothesis:** Models that remove the daily pattern explicitly will be more accurate than models that treat the raw series as if its level were stable. SARIMAX may help if renewable share itself can be forecast accurately. We test these claims on unseen data rather than judging in-sample fit.

## 9. Forecast design

*Use one chronological split and prevent future information from entering training*

We use 180 training days, 7 validation days, and one untouched 7-day test set. Every model predicts all 168 test hours recursively or directly. The data is never shuffled.

In [7]:
bundle = run_forecasts(REGION, data)
split_table = pd.DataFrame({
    "Split": ["Training", "Validation", "Final test"],
    "Start": [bundle.train.index.min(), bundle.validation.index.min(), bundle.test.index.min()],
    "End": [bundle.train.index.max(), bundle.validation.index.max(), bundle.test.index.max()],
    "Hours": [len(bundle.train), len(bundle.validation), len(bundle.test)],
})
display(split_table)

split_frame = bundle.frame.reset_index()
fig = px.line(split_frame, x="datetime", y="carbon", title=f"{REGION}: chronological model window")
fig.add_vrect(x0=bundle.validation.index.min(), x1=bundle.validation.index.max(),
              fillcolor="#facc15", opacity=0.2, line_width=0, annotation_text="Validation")
fig.add_vrect(x0=bundle.test.index.min(), x1=bundle.test.index.max(),
              fillcolor="#ef4444", opacity=0.2, line_width=0, annotation_text="Test")
fig.update_yaxes(title="Carbon intensity (gCO₂eq/kWh)")
show_chart(fig)

display(Markdown(
    f"All **{len(bundle.test)} test hours** occur after training and validation. "
    "The same timestamps and actual values are used for every score, so the comparison is fair."
))

,Split,Start,End,Hours
0,Training,2026-02-20 16:00:00+00:00,2026-08-19 15:00:00+00:00,4320
1,Validation,2026-08-19 16:00:00+00:00,2026-08-26 15:00:00+00:00,168
2,Final test,2026-08-26 16:00:00+00:00,2026-09-02 15:00:00+00:00,168


All **168 test hours** occur after training and validation. The same timestamps and actual values are used for every score, so the comparison is fair.

## 10. Baseline forecasts

*Set simple targets that advanced models must beat*

The naïve baseline repeats the last observed value. The seasonal-naïve baseline repeats the latest 24-hour pattern.

In [8]:
baseline = bundle.metrics[bundle.metrics["Model"].str.startswith("Baseline")]
display(baseline.round(3))
baseline_names = baseline["Model"].tolist()
plot_data = bundle.forecasts[["Actual", *baseline_names]].reset_index().melt(
    id_vars="datetime", var_name="Series", value_name="Carbon intensity")
fig = px.line(plot_data, x="datetime", y="Carbon intensity", color="Series",
              title=f"{REGION}: baseline forecasts on the unseen week")
show_chart(fig)
best_baseline = baseline.sort_values("RMSE").iloc[0]
display(Markdown(
    f"**{best_baseline['Model']}** is the stronger baseline with RMSE "
    f"**{best_baseline['RMSE']:.2f} gCO₂eq/kWh**. A useful advanced model should improve on it."
))

,Rank,Model,MAE,RMSE,MAPE (%),sMAPE (%),R²,Fit time (s),RMSE improvement vs best baseline (%)
4,5,Baseline: seasonal naive (24h),15.565,19.353,3.427,3.463,0.945,0.0,0.000
9,10,Baseline: naive last value,69.256,93.087,18.345,15.611,-0.282,0.0,-380.994


**Baseline: seasonal naive (24h)** is the stronger baseline with RMSE **19.35 gCO₂eq/kWh**. A useful advanced model should improve on it.

## 11. ARIMA, SARIMA, SARIMAX, and differencing models

*Compare the four required stationary/non-stationary combinations*

“Stationary” models receive an external lag-24 difference and are reconstructed to the original scale. “Non-stationary” models receive raw levels and estimate differencing internally. ARIMA has no explicit seasonal terms. SARIMA adds a 24-hour seasonal structure.

In [9]:
required_names = [
    "ARIMA stationary: (2,0,1) on Δ24",
    "ARIMA non-stationary: (2,1,1)",
    "SARIMA stationary: (1,0,1)(1,0,1,24) on Δ24",
    "SARIMA non-stationary: (1,0,1)(1,1,1,24)",
]
required = bundle.metrics[bundle.metrics["Model"].isin(required_names)].copy().sort_values("RMSE")
required["Input"] = np.where(required["Model"].str.contains("stationary:.*Δ24", regex=True),
                             "Stationary lag-24 difference", "Raw/non-stationary level")
required["Seasonal model"] = np.where(required["Model"].str.startswith("SARIMA"), "Yes", "No")
display(required[["Rank", "Model", "Input", "Seasonal model", "MAE", "RMSE", "MAPE (%)", "R²", "Fit time (s)"]].round(3))

long_required = bundle.forecasts[["Actual", *required_names]].reset_index().melt(
    id_vars="datetime", var_name="Series", value_name="Carbon intensity")
fig = px.line(long_required, x="datetime", y="Carbon intensity", color="Series",
              title=f"{REGION}: four required ARIMA/SARIMA forecasts")
show_chart(fig)

best_required = required.iloc[0]
best_arima = required[required["Seasonal model"] == "No"]["RMSE"].min()
best_sarima = required[required["Seasonal model"] == "Yes"]["RMSE"].min()
stationary_best = required[required["Input"] == "Stationary lag-24 difference"]["RMSE"].min()
raw_best = required[required["Input"] == "Raw/non-stationary level"]["RMSE"].min()
seasonal_effect = (best_sarima - best_arima) / best_arima * 100
differencing_effect = (raw_best - stationary_best) / stationary_best * 100
display(Markdown(
    f"The best required combination is **{best_required['Model']}** with RMSE "
    f"**{best_required['RMSE']:.2f}**. The best SARIMA has **{abs(seasonal_effect):.1f}% "
    f"{'higher' if seasonal_effect > 0 else 'lower'}** RMSE than the best ARIMA. The best raw-input model has "
    f"**{abs(differencing_effect):.1f}% {'higher' if differencing_effect > 0 else 'lower'}** RMSE than the best "
    "explicitly differenced model. For this test week, the held-out result decides which complexity is useful."
))

,Rank,Model,Input,Seasonal model,MAE,RMSE,MAPE (%),R²,Fit time (s)
0,1,"SARIMA stationary: (1,0,1)(1,0,1,24) on Δ24",Stationary lag-24 difference,Yes,13.706,17.211,3.117,0.956,3.513
1,2,"SARIMA non-stationary: (1,0,1)(1,1,1,24)",Raw/non-stationary level,Yes,13.731,17.238,3.122,0.956,11.795
3,4,"ARIMA stationary: (2,0,1) on Δ24",Stationary lag-24 difference,No,14.784,18.601,3.305,0.949,0.148
10,11,"ARIMA non-stationary: (2,1,1)",Raw/non-stationary level,No,69.550,95.888,18.608,-0.361,0.255


The best required combination is **SARIMA stationary: (1,0,1)(1,0,1,24) on Δ24** with RMSE **17.21**. The best SARIMA has **7.5% lower** RMSE than the best ARIMA. The best raw-input model has **0.2% higher** RMSE than the best explicitly differenced model. For this test week, the held-out result decides which complexity is useful.

### 11.1 SARIMAX with renewable percentage

*Add renewable share without revealing the future*

The future renewable percentage is unknown at forecast time. We therefore forecast it with the last observed 24-hour pattern before supplying it to SARIMAX. This is harder but fairer than using the true future renewable values.

In [10]:
sarimax = bundle.metrics[bundle.metrics["Model"] == "SARIMAX + forecast renewable %"].iloc[0]
display(pd.DataFrame([sarimax]).round(3))
display(Markdown(
    f"Leakage-safe SARIMAX gives RMSE **{sarimax['RMSE']:.2f}** and R² "
    f"**{sarimax['R²']:.3f}**. Its value depends on two forecasts: renewable share first, then carbon intensity. "
    "A strong historical correlation alone does not guarantee the best future forecast."
))

,Rank,Model,MAE,RMSE,MAPE (%),sMAPE (%),R²,Fit time (s),RMSE improvement vs best baseline (%)
6,7,SARIMAX + forecast renewable %,15.698,19.421,3.468,3.499,0.944,6.888,-0.35


Leakage-safe SARIMAX gives RMSE **19.42** and R² **0.944**. Its value depends on two forecasts: renewable share first, then carbon intensity. A strong historical correlation alone does not guarantee the best future forecast.

## 12. Holt-Winters models

*Update level, daily seasonality, and an optional damped trend*

Both models use additive 24-hour seasonality. One has no trend. The other includes a trend that gradually weakens over the forecast horizon.

In [11]:
holt = bundle.metrics[bundle.metrics["Model"].str.startswith("Holt-Winters")].sort_values("RMSE")
display(holt.round(3))
best_holt = holt.iloc[0]
display(Markdown(
    f"**{best_holt['Model']}** is the stronger Holt-Winters choice, with RMSE "
    f"**{best_holt['RMSE']:.2f}**. This family is fast and easy to explain, which makes it a useful operational benchmark."
))

,Rank,Model,MAE,RMSE,MAPE (%),sMAPE (%),R²,Fit time (s),RMSE improvement vs best baseline (%)
5,6,Holt-Winters damped trend,15.440,19.391,3.404,3.482,0.944,0.099,-0.194
7,8,Holt-Winters additive seasonal,15.545,19.510,3.422,3.503,0.944,0.189,-0.810


**Holt-Winters damped trend** is the stronger Holt-Winters choice, with RMSE **19.39**. This family is fast and easy to explain, which makes it a useful operational benchmark.

## 13. Machine-learning and deep-learning models

*Learn non-linear relationships from past values and calendar features*

Random Forest and the MLP neural network use lags 1, 2, 24, 48, and 168, rolling statistics, hour of day, and day of week. Rolling features are shifted by one hour, so they never include the target being predicted.

In [12]:
learned = bundle.metrics[bundle.metrics["Model"].str.startswith(("ML:", "DL:"))].sort_values("RMSE")
display(learned.round(3))
best_learned = learned.iloc[0]
display(pd.DataFrame({"Feature used": FEATURES}))
display(Markdown(
    f"**{best_learned['Model']}** is the better learned model with RMSE "
    f"**{best_learned['RMSE']:.2f}**. Both models make a true recursive 168-hour forecast, so later predictions use earlier predictions rather than hidden actual values."
))

,Rank,Model,MAE,RMSE,MAPE (%),sMAPE (%),R²,Fit time (s),RMSE improvement vs best baseline (%)
2,3,ML: Random Forest,14.149,17.570,3.198,3.206,0.954,3.297,9.215
8,9,DL: MLP neural network,29.674,36.245,6.522,6.625,0.806,1.260,-87.284


,Feature used
0,lag_1
1,lag_2
2,lag_24
3,lag_48
4,lag_168
5,roll_mean_24
6,roll_std_24
7,roll_mean_168
8,hour_sin
9,hour_cos


**ML: Random Forest** is the better learned model with RMSE **17.57**. Both models make a true recursive 168-hour forecast, so later predictions use earlier predictions rather than hidden actual values.

## 14. Forecast accuracy and complete model comparison

*Rank every model using the same unseen week*

MAE is the typical absolute miss. RMSE penalizes large misses more strongly. MAPE and sMAPE express relative error. R² shows how much test variation is explained. Lower error and higher R² are better.

In [13]:
scores = bundle.metrics.copy()
display(scores.round(3))
fig = px.bar(scores.sort_values("RMSE"), x="RMSE", y="Model", orientation="h", color="RMSE",
             color_continuous_scale="RdYlGn_r", title=f"{REGION}: model ranking by test RMSE")
fig.update_layout(yaxis={"categoryorder": "total descending"}, height=620)
show_chart(fig)

winner = scores.iloc[0]
display(Markdown(
    f"The overall winner is **{winner['Model']}**, with RMSE **{winner['RMSE']:.2f}**, "
    f"MAE **{winner['MAE']:.2f}**, and a **{winner['RMSE improvement vs best baseline (%)']:.1f}%** RMSE improvement "
    "over the strongest baseline. This conclusion applies to the shared final week; rolling-origin tests would strengthen a production decision."
))

,Rank,Model,MAE,RMSE,MAPE (%),sMAPE (%),R²,Fit time (s),RMSE improvement vs best baseline (%)
0,1,"SARIMA stationary: (1,0,1)(1,0,1,24) on Δ24",13.706,17.211,3.117,3.131,0.956,3.513,11.070
1,2,"SARIMA non-stationary: (1,0,1)(1,1,1,24)",13.731,17.238,3.122,3.136,0.956,11.795,10.929
2,3,ML: Random Forest,14.149,17.570,3.198,3.206,0.954,3.297,9.215
3,4,"ARIMA stationary: (2,0,1) on Δ24",14.784,18.601,3.305,3.338,0.949,0.148,3.885
4,5,Baseline: seasonal naive (24h),15.565,19.353,3.427,3.463,0.945,0.000,0.000
5,6,Holt-Winters damped trend,15.440,19.391,3.404,3.482,0.944,0.099,-0.194
6,7,SARIMAX + forecast renewable %,15.698,19.421,3.468,3.499,0.944,6.888,-0.350
7,8,Holt-Winters additive seasonal,15.545,19.510,3.422,3.503,0.944,0.189,-0.810
8,9,DL: MLP neural network,29.674,36.245,6.522,6.625,0.806,1.260,-87.284
9,10,Baseline: naive last value,69.256,93.087,18.345,15.611,-0.282,0.000,-380.994


The overall winner is **SARIMA stationary: (1,0,1)(1,0,1,24) on Δ24**, with RMSE **17.21**, MAE **13.71**, and a **11.1%** RMSE improvement over the strongest baseline. This conclusion applies to the shared final week; rolling-origin tests would strengthen a production decision.

### 14.1 Actual values versus the strongest forecasts

*Check whether the numerical winner also follows the visible hourly shape*

The chart shows the overall winner, the best required ARIMA/SARIMA combination, and the strongest baseline.

In [14]:
best_model = scores.iloc[0]["Model"]
best_required_name = required.iloc[0]["Model"]
best_baseline_name = baseline.sort_values("RMSE").iloc[0]["Model"]
selected = list(dict.fromkeys(["Actual", best_model, best_required_name, best_baseline_name]))
selected_long = bundle.forecasts[selected].reset_index().melt(
    id_vars="datetime", var_name="Series", value_name="Carbon intensity")
fig = px.line(selected_long, x="datetime", y="Carbon intensity", color="Series",
              title=f"{REGION}: actual values and selected forecasts")
show_chart(fig)
display(Markdown(
    "A good model should follow both the daily peaks and the midday lows. Large vertical gaps show "
    "hours where the model misses the level even if it keeps the general shape."
))

A good model should follow both the daily peaks and the midday lows. Large vertical gaps show hours where the model misses the level even if it keeps the general shape.

## 15. Residual analysis

*Check bias, spread, normality, and remaining time dependence*

A residual is actual minus forecast. A well-calibrated point forecast should have residuals centered near zero with little repeated structure. Ljung-Box p-values below 0.05 indicate remaining autocorrelation.

In [15]:
diagnostics = bundle.diagnostics.copy()
display(diagnostics.round(4))
winner_residual = bundle.forecasts["Actual"] - bundle.forecasts[best_model]
residual_acf = acf(winner_residual, nlags=72, fft=True)
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=("Residuals over time", "Residual histogram",
                                    "Residual ACF", "Actual versus residual"))
fig.add_trace(go.Scatter(x=winner_residual.index, y=winner_residual, mode="lines",
                         line=dict(color="#2563eb"), showlegend=False), row=1, col=1)
fig.add_trace(go.Histogram(x=winner_residual, marker_color="#7c3aed", showlegend=False), row=1, col=2)
fig.add_trace(go.Bar(x=np.arange(73), y=residual_acf, marker_color="#dc2626", showlegend=False), row=2, col=1)
fig.add_trace(go.Scatter(x=bundle.forecasts["Actual"], y=winner_residual, mode="markers",
                         marker=dict(color="#0f766e", opacity=0.65), showlegend=False), row=2, col=2)
fig.update_layout(height=760, title=f"{REGION}: residual diagnostics for {best_model}")
show_chart(fig)

winner_diag = diagnostics[diagnostics["Model"] == best_model].iloc[0]
display(Markdown(
    f"The winning model has mean residual **{winner_diag['Mean residual (bias)']:.2f}** and "
    f"lag-24 Ljung-Box p-value **{winner_diag['Ljung-Box p (lag 24)']:.4g}**. "
    f"It **{'passes' if winner_diag['White-noise at 5%?'] == 'Yes' else 'does not pass'}** the 5% white-noise check. "
    "Remaining daily structure means the model is useful but not perfect, and its uncertainty should not be ignored."
))

,Model,Mean residual (bias),Residual SD,Ljung-Box p (lag 24),White-noise at 5%?,Jarque-Bera p,Rank
4,"SARIMA stationary: (1,0,1)(1,0,1,24) on Δ24",4.3212,16.7093,0.0,No,0.7224,1
5,"SARIMA non-stationary: (1,0,1)(1,1,1,24)",4.3862,16.7205,0.0,No,0.7228,2
9,ML: Random Forest,3.1540,17.3359,0.0,No,0.3646,3
2,"ARIMA stationary: (2,0,1) on Δ24",5.5336,17.8121,0.0,No,0.7573,4
1,Baseline: seasonal naive (24h),6.6488,18.2295,0.0,No,0.7673,5
8,Holt-Winters damped trend,11.4800,15.6737,0.0,No,0.5361,6
6,SARIMAX + forecast renewable %,6.2506,18.4424,0.0,No,0.8215,7
7,Holt-Winters additive seasonal,11.7709,15.6055,0.0,No,0.4846,8
10,DL: MLP neural network,9.0231,35.2090,0.0,No,0.1451,9
0,Baseline: naive last value,-43.6845,82.4460,0.0,No,0.0000,10


The winning model has mean residual **4.32** and lag-24 Ljung-Box p-value **7.713e-292**. It **does not pass** the 5% white-noise check. Remaining daily structure means the model is useful but not perfect, and its uncertainty should not be ignored.

## 16. Research conclusion

*Answer the research question using held-out evidence*

The data contains a strong daily cycle, a slower long-term change, and a close inverse relationship with renewable share. A lag-24 difference gives a more stable series for forecasting. The final model choice is based on unseen error, not visual fit or training accuracy.

In [16]:
best_required = required.iloc[0]
winner_diag = diagnostics[diagnostics["Model"] == best_model].iloc[0]
conclusion = f'''
**Final answer for {REGION}:** The best model in the common seven-day test is **{best_model}**.
It achieves RMSE **{winner['RMSE']:.2f} gCO₂eq/kWh** and improves on the best simple baseline by
**{winner['RMSE improvement vs best baseline (%)']:.1f}%**. Among the four required combinations,
**{best_required['Model']}** is best with RMSE **{best_required['RMSE']:.2f}**.

**Practical meaning:** Daily differencing is especially important when the raw level drifts. A seasonal model is
helpful only when its extra terms reduce unseen error. Renewable percentage is useful explanatory information,
but a real forecast must predict renewable percentage too. The residual test shows that some repeated structure
remains, so forecasts should be refreshed often and used with uncertainty bands in operational decisions.
'''
display(Markdown(conclusion))


**Final answer for IN-SO:** The best model in the common seven-day test is **SARIMA stationary: (1,0,1)(1,0,1,24) on Δ24**.
It achieves RMSE **17.21 gCO₂eq/kWh** and improves on the best simple baseline by
**11.1%**. Among the four required combinations,
**SARIMA stationary: (1,0,1)(1,0,1,24) on Δ24** is best with RMSE **17.21**.

**Practical meaning:** Daily differencing is especially important when the raw level drifts. A seasonal model is
helpful only when its extra terms reduce unseen error. Renewable percentage is useful explanatory information,
but a real forecast must predict renewable percentage too. The residual test shows that some repeated structure
remains, so forecasts should be refreshed often and used with uncertainty bands in operational decisions.


## 17. Individual contribution

*Document the work completed by Shantharam P*

| Work area | Individual contribution | Evidence |
|---|---|---|
| Problem framing | Defined the regional forecast question and the carbon-aware use case | Sections 1 and 16 |
| Data engineering | Built API pagination, aligned two signals, enforced a common timeline, and checked gaps and duplicates | Section 3 and `scripts/fetch_data.py` |
| Exploratory analysis | Produced interactive trend, seasonality, stationarity, ACF, and PACF analysis | Sections 4–8 |
| Statistical models | Implemented the four required ARIMA/SARIMA combinations, SARIMAX, and Holt-Winters | Sections 11–12 |
| ML and DL | Built leakage-safe lag features, Random Forest, and an MLP neural network | Section 13 |
| Evaluation | Used a chronological split, five accuracy measures, model ranking, and residual diagnostics | Sections 9, 14, and 15 |
| Reproducibility | Centralized shared code, fixed random seeds, saved comparison outputs, and documented dependencies | `src/`, `scripts/`, `results/`, and `requirements.txt` |
| Communication | Wrote simple-English interpretations and linked results to practical carbon-aware decisions | All sections |

This notebook records Shantharam P as the contributor responsible for the IN-SO regional analysis.

## 18. Reproducibility, limitations, and references

*What another reader needs to rerun and interpret the study correctly*

Run all cells from the repository root after installing `requirements.txt`. The included CSV files already use the common timeline. To refresh them, set `ELECTRICITYMAPS_API_KEY` and run `python scripts/fetch_data.py --zones IN-NO IN-WE IN-SO --overwrite`.

**Limitations:** Results come from one final seven-day test period. Electricity Maps may estimate some historical observations. SARIMAX depends on a separate renewable-share forecast. The models predict regional average carbon intensity, not a specific power plant or consumer contract.

**Sources:** [Electricity Maps API](https://portal.electricitymaps.com/docs/getting-started), [statsmodels time-series documentation](https://www.statsmodels.org/stable/tsa.html), and [scikit-learn documentation](https://scikit-learn.org/stable/).